This notebook walks through the pipeline interactively:
1. Load data -> 2. Validate -> 3–5. Engineer features & split -> 6. Train -> 7. Evaluate -> 8. Report

> **Note**: for the statistical benchmarking use `benchmark_wilcoxon.ipynb`.

## Step 0 — Setup

Import libraries and set up the environment.

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is on path
PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from config import *

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

print('Environment ready.')
print(f'  pandas: {pd.__version__}')
print(f'  numpy:  {np.__version__}')
import xgboost; print(f'  xgboost: {xgboost.__version__}')
import sklearn; print(f'  sklearn: {sklearn.__version__}')

Environment ready.
  pandas: 3.0.6
  numpy:  2.4.6


  xgboost: 3.2.0
  sklearn: 1.9.1


## Step 1 — Load Data

If real DOH + Project CCHAIN data exists at `data/synthetic_dengue_data.csv`, it loads that. Otherwise, it generates a synthetic panel of 936 city-weeks (3 cities × 6 years × 52 weeks) with realistic seasonal dengue dynamics and COVID-era disruption.

The panel contains:
- `city` — Davao City, Cagayan de Oro City, Zamboanga City
- `year`, `week_number`, `date` — temporal keys
- `weekly_dengue_cases` — the target variable
- `rainfall`, `temperature`, `tmin`, `tmax`, `humidity` — meteorological inputs
- `heat_index`, `diurnal_temp_range`, `ndvi`, `spi3`, `pnp` — bioclimatic indicators

In [2]:
# Prefer the real per-city panels in datasets/; fall back to synthetic.
try:
    from src.data_loader import load_real_panel
    df = load_real_panel()
except Exception as exc:  # noqa: BLE001
    print(f'Real panel unavailable ({exc}); generating synthetic data.')
    from data.generate_synthetic_data import generate_panel, add_derived_weather
    df = add_derived_weather(generate_panel())

print(f'Loaded {len(df)} rows, {len(df.columns)} columns')
print(f'Cities: {df["city"].unique().tolist()}')
print(f'Year range: {df["year"].min()}-{df["year"].max()}')
df.head(10)

Loading real panels from: C:\Users\princ\miniconda3\envs\forecasting\datasets
  read dengue_davao_city_week_panel_2016_2021.csv: 262 rows
  read dengue_cagayan_de_oro_city_week_panel_2016_2021.csv: 262 rows
  read dengue_zamboanga_city_week_panel_2016_2021.csv: 262 rows
  merged: 786 rows, 37 columns
  Dropping 3 row(s) with week_number > 52
  Collapsing 6 duplicate city-year-week rows (median of reported values)
  Dropping 9 row(s) with missing weekly_dengue_cases
  Cagayan de Oro City: winsorising 2 week(s) above cap 484 (max 998)
  Zamboanga City: winsorising 1 week(s) above cap 513 (max 3382)
  final: 771 rows, 3 cities, years 2016.0-2020.0
Loaded 771 rows, 38 columns
Cities: ['Cagayan de Oro City', 'Davao City', 'Zamboanga City']
Year range: 2016.0-2020.0


,year,week_number,weekly_dengue_cases,rainfall,temperature,humidity,cases_lag1,cases_lag2,cases_lag4,cases_rollmean4,...,diurnal_temp_range,spi3,pnp,ndvi,sin_week,cos_week,city,data_quality_flags,source_file,date
0,2016.00,1.00,64.00,1.08,25.54,85.07,NaN,NaN,NaN,NaN,...,6.05,-1.26,12.81,0.71,0.12,0.99,Cagayan de Oro City,insufficient_history,dengue_cagayan_de_oro_city_week_panel_2016_202...,2016-01-04
1,2016.00,2.00,46.00,2.99,25.67,81.84,66.00,NaN,NaN,NaN,...,7.13,-1.26,12.81,0.70,0.24,0.97,Cagayan de Oro City,insufficient_history,dengue_cagayan_de_oro_city_week_panel_2016_202...,2016-01-11
2,2016.00,3.00,55.00,5.42,25.89,81.42,50.00,66.00,NaN,NaN,...,6.29,-1.26,12.81,0.68,0.35,0.94,Cagayan de Oro City,insufficient_history,dengue_cagayan_de_oro_city_week_panel_2016_202...,2016-01-18
3,2016.00,4.00,54.00,8.32,25.57,77.84,64.00,50.00,NaN,56.50,...,7.29,-1.26,12.81,0.65,0.46,0.89,Cagayan de Oro City,insufficient_history,dengue_cagayan_de_oro_city_week_panel_2016_202...,2016-01-25
4,2016.00,5.00,38.00,6.12,25.96,84.70,46.00,64.00,66.00,53.75,...,5.59,-1.58,31.81,0.63,0.57,0.82,Cagayan de Oro City,ok,dengue_cagayan_de_oro_city_week_panel_2016_202...,2016-02-01
5,2016.00,6.00,36.00,10.71,25.86,82.42,55.00,46.00,50.00,54.75,...,5.50,-1.58,31.81,0.61,0.66,0.75,Cagayan de Oro City,ok,dengue_cagayan_de_oro_city_week_panel_2016_202...,2016-02-08
6,2016.00,7.00,32.00,6.57,25.56,81.76,54.00,55.00,64.00,48.25,...,6.55,-1.58,31.81,0.60,0.75,0.66,Cagayan de Oro City,ok,dengue_cagayan_de_oro_city_week_panel_2016_202...,2016-02-15
7,2016.00,8.00,34.00,7.89,25.74,82.95,38.00,54.00,46.00,45.75,...,6.08,-1.58,31.81,0.59,0.82,0.57,Cagayan de Oro City,ok,dengue_cagayan_de_oro_city_week_panel_2016_202...,2016-02-22
8,2016.00,9.00,37.00,0.00,26.47,84.36,36.00,38.00,55.00,40.00,...,5.86,-2.18,15.25,0.58,0.89,0.46,Cagayan de Oro City,ok,dengue_cagayan_de_oro_city_week_panel_2016_202...,2016-02-29
9,2016.00,10.00,24.00,5.80,26.51,81.29,32.00,36.00,54.00,35.00,...,6.66,-2.28,12.49,0.56,0.94,0.35,Cagayan de Oro City,ok,dengue_cagayan_de_oro_city_week_panel_2016_202...,2016-03-07


## Step 2 — Validate Data

Checks from proposal §5.2:
- Schema completeness (all required columns present)
- Missing-week check (each city-year should have 52 weeks)
- Plausibility review (non-negative cases, sane weather bounds)
- Missing-value rates below 5% threshold

In [3]:
from src.features.validate_data import validate_data
report = validate_data(df)
print(json.dumps(report, indent=2, default=str))

[OK] All data quality checks passed.
  Validation report saved — C:\Users\princ\miniconda3\envs\forecasting\outputs\data_validation_report.json
{
  "status": "PASSED_WITH_WARNINGS",
  "n_rows": 771,
  "n_cities": 3,
  "years": [
    2016.0,
    2017.0,
    2018.0,
    2019.0,
    2020.0
  ],
  "total_missing_cells": 189,
  "missing_pct": 0.65,
  "city_breakdown": {
    "Cagayan de Oro City": 257,
    "Davao City": 257,
    "Zamboanga City": 257
  },
  "plausibility_warnings": [
    "Missing weeks: Cagayan de Oro City 2016.0 has 51 weeks (expected 52)",
    "Missing weeks: Cagayan de Oro City 2017.0 has 51 weeks (expected 52)",
    "Missing weeks: Cagayan de Oro City 2018.0 has 51 weeks (expected 52)",
    "Missing weeks: Davao City 2016.0 has 51 weeks (expected 52)",
    "Missing weeks: Davao City 2017.0 has 51 weeks (expected 52)",
    "Missing weeks: Davao City 2018.0 has 51 weeks (expected 52)",
    "Missing weeks: Zamboanga City 2016.0 has 51 weeks (expected 52)",
    "Missing week

## Step 3–5 — Engineer Features & Temporal Split

Builds all **28 features** (proposal §5.6) and the **t+2 target**. Applies the leakage-free temporal split:
- **Train**: 2016–2019 (~70% of panel)
- **Embargo**: first 2 weeks of 2020 (excluded)
- **Test**: 2020–2021 (~30%, held-out)


In [4]:
from src.features.build_features import build_feature_matrix
X, y, city, splits, dates = build_feature_matrix(df)

print(f'Feature matrix: {X.shape[0]} rows × {X.shape[1]} features')
print(f'\nFeature groups:')
print(f'  Weather lags (12): {[f for f in WEATHER_LAGS]}')
print(f'  Rolling aggregates (3): {ROLLING_AGGREGATES}')
print(f'  Case momentum (4): {CASE_MOMENTUM}')
print(f'  Bioclimatic (7): {BIOCLIMATIC}')
print(f'  Seasonality (2): {SEASONALITY}')

print(f'\nTemporal split:')
for k, v in splits.items():
    print(f'  {k:10s}: {v.sum()} rows')

Feature matrix: 753 rows x 28 features
  Features: 28 (expected 28)
  Train:      606 rows  (2016–2019)
  Embargo:    3 rows  (skipped)
  Test (2020–2021): 144 rows
Feature matrix: 753 rows × 28 features

Feature groups:
  Weather lags (12): ['rainfall_lag1', 'rainfall_lag2', 'rainfall_lag3', 'rainfall_lag4', 'temperature_lag1', 'temperature_lag2', 'temperature_lag3', 'temperature_lag4', 'humidity_lag1', 'humidity_lag2', 'humidity_lag3', 'humidity_lag4']
  Rolling aggregates (3): ['rainfall_rollsum4', 'temperature_rollmean4', 'humidity_rollmean4']
  Case momentum (4): ['cases_lag1', 'cases_lag2', 'cases_lag4', 'cases_rollmean4']
  Bioclimatic (7): ['heat_index', 'tmin', 'tmax', 'diurnal_temp_range', 'spi3', 'pnp', 'ndvi']
  Seasonality (2): ['sin_week', 'cos_week']

Temporal split:
  train     : 606 rows
  embargo   : 3 rows
  test      : 144 rows


In [5]:
# Visualise the temporal split using the dates returned by
# build_feature_matrix (aligned with X/y/splits).
split_df = pd.DataFrame({
    'date': pd.to_datetime(pd.Series(dates.values)),
    'cases': y.values,
    'city': city.values,
    'set': 'other',
})
split_df.loc[splits['train'], 'set'] = 'train'
split_df.loc[splits['embargo'], 'set'] = 'embargo'
split_df.loc[splits['test'], 'set'] = 'test'

colors = {'train': '#4e79a7', 'embargo': '#f28e2b', 'test': '#59a14f'}
fig, ax = plt.subplots(figsize=(14, 3.2))
for set_name in ['train', 'embargo', 'test']:
    sub = split_df[split_df['set'] == set_name]
    if len(sub):
        ax.scatter(sub['date'], [1] * len(sub), s=2,
                   c=colors[set_name], label=set_name.capitalize(), alpha=0.6)
ax.set_yticks([])
ax.set_xlabel('Forecast-origin date')
ax.set_title('Temporal Split: Train (2016–2019) / Test (2020–2021)')
ax.legend(loc='upper right')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# Feature correlation heatmap (subset for readability)
corr = X[CASE_MOMENTUM + ['rainfall_lag1', 'temperature_lag1',
                          'humidity_lag1', 'sin_week', 'cos_week']].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, square=True)
ax.set_title('Feature Correlation Heatmap (subset)')
plt.tight_layout()
plt.show()

## Step 6 — Train All Models

Trains 4 models with forward-chaining CV for hyperparameter tuning (training data only):
1. **Persistence** — baseline: forecast = most recent case count
2. **Case-only XGBoost** — 6 case/seasonality features
3. **Full XGBoost** — all 28 features


In [6]:
from src.models.train import train_all_models
results = train_all_models(X, y, splits)

TRAINING MODELS (leakage-free temporal design)
  Train: 2016–2019  |  Test: 2020–2021  |  Embargo: 2 weeks

  Training: persistence
    Train MAE=26.2, RMSE=49.2, R²=0.450
    Test  MAE=6.0, RMSE=9.3, R²=0.188

  Training: xgb_case_only


  CV tuning complete. Best: depth=3, lr=0.05, n_est=40, shrinkage=0.4, CV MAE=22.58
    Train MAE=23.5, RMSE=46.5, R²=0.509
    Test  MAE=6.0, RMSE=8.9, R²=0.256

  Training: xgb_full


  CV tuning complete. Best: depth=3, lr=0.05, n_est=40, shrinkage=0.4, CV MAE=22.81
    Train MAE=23.3, RMSE=46.5, R²=0.509
    Test  MAE=5.9, RMSE=8.8, R²=0.271


## Step 7 — Evaluate (2020–2021 Test Set)

Computes MAE, RMSE, R² on the held-out 2020–2021 test set with block-bootstrap 95% confidence intervals. Also shows per-city breakdown.


In [7]:
from src.evaluation.evaluate import evaluate_all_models
report = evaluate_all_models(results, splits, X, y, city)


EVALUATION (held-out test set)

  persistence


    Test MAE=6.0 (CI: 4.5–7.5), RMSE=9.3, R²=0.188
      Cagayan de Oro City       MAE=6.5  RMSE=9.6  R²=-0.280
      Davao City                MAE=7.9  RMSE=11.0  R²=-0.142
      Zamboanga City            MAE=3.6  RMSE=6.6  R²=0.302

  xgb_case_only


    Test MAE=6.0 (CI: 4.6–7.4), RMSE=8.9, R²=0.256
      Cagayan de Oro City       MAE=6.4  RMSE=9.0  R²=-0.139
      Davao City                MAE=7.4  RMSE=10.4  R²=-0.025
      Zamboanga City            MAE=4.3  RMSE=6.7  R²=0.285

  xgb_full


    Test MAE=5.9 (CI: 4.6–7.3), RMSE=8.8, R²=0.271
      Cagayan de Oro City       MAE=6.3  RMSE=8.9  R²=-0.109
      Davao City                MAE=7.0  RMSE=10.0  R²=0.053
      Zamboanga City            MAE=4.4  RMSE=7.1  R²=0.192

----------------------------------------
WEATHER ABLATION (case-only vs full XGBoost)
----------------------------------------
  Case-only MAE: 6.0 (CI: 4.6–7.4)
  Full MAE:      5.9 (CI: 4.6–7.3)
  MAE improvement: 0.1 (+1.0%)

  Case-only RMSE: 8.9 (CI: 6.6–10.8)
  Full RMSE:      8.8 (CI: 6.5–10.7)
  RMSE improvement: 0.1 (+1.0%)

  Persistence MAE: 6.0
  Full XGBoost relative MAE (full/persist): 0.989
  Case-only XGBoost relative MAE: 0.999

Full evaluation report saved → C:\Users\princ\miniconda3\envs\forecasting\outputs\results.json


In [8]:
# Summary table
rows = []
for name in report['models']:
    test = report['models'][name]['test']['pooled']
    ci = report['models'][name]['test']['bootstrap_ci']
    rows.append({
        'Model': name,
        'Test MAE': f"{test['MAE']:.1f}",
        'MAE 95% CI': f"[{ci['MAE_ci'][0]:.1f}, {ci['MAE_ci'][1]:.1f}]",
        'Test RMSE': f"{test['RMSE']:.1f}",
        'Test R²': f"{test['R2']:.3f}",
    })
summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUTPUT_DIR / 'summary_table.csv', index=False)
summary_df

,Model,Test MAE,MAE 95% CI,Test RMSE,Test R²
0,persistence,6.0,"[4.5, 7.5]",9.3,0.188
1,xgb_case_only,6.0,"[4.6, 7.4]",8.9,0.256
2,xgb_full,5.9,"[4.6, 7.3]",8.8,0.271


In [9]:
# Per-city MAE comparison
fig, ax = plt.subplots(figsize=(12, 5))
models = list(report['models'].keys())
cities_list = sorted(set().union(*[report['models'][m]['test']['per_city'].keys() for m in models]))
x = np.arange(len(cities_list))
width = 0.2
colors = ['#4e79a7', '#f28e2b', '#e15759', '#59a14f']
for i, model_name in enumerate(models):
    maes = [report['models'][model_name]['test']['per_city'].get(c, {}).get('MAE', 0) for c in cities_list]
    ax.bar(x + i * width, maes, width, label=model_name, color=colors[i])
ax.set_xticks(x + width * (len(models) - 1) / 2)
ax.set_xticklabels([c.replace(' City', '').replace(' de Oro', ' de Oro') for c in cities_list], rotation=15)
ax.set_ylabel('MAE (cases)')
ax.set_title('Per-City MAE — 2020–2021 Test Year')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [10]:
# Per-city performance breakdown with percentage improvements
print("=" * 80)
print("PER-CITY PERFORMANCE BREAKDOWN (2020–2021 Test Year)")
print("=" * 80)

per_city_rows = []
for model_name in report["models"]:
    pc_data = report["models"][model_name]["test"]["per_city"]
    for city_name in sorted(pc_data.keys()):
        d = pc_data[city_name]
        per_city_rows.append({
            "City": city_name,
            "Model": model_name,
            "MAE": round(d["MAE"], 1),
            "RMSE": round(d["RMSE"], 1),
            "R2": round(d["R2"], 3),
        })

per_city_df = pd.DataFrame(per_city_rows)

# Pivots for readability
pivot_mae = per_city_df.pivot(index="City", columns="Model", values="MAE")
pivot_rmse = per_city_df.pivot(index="City", columns="Model", values="RMSE")
pivot_r2 = per_city_df.pivot(index="City", columns="Model", values="R2")

print("MAE by City x Model:")
print(pivot_mae.to_string())
print()
print("RMSE by City x Model:")
print(pivot_rmse.to_string())
print()
print("R2 by City x Model:")
print(pivot_r2.to_string())
print()

# Percentage improvement from case-only to full per city
print("Weather ablation improvement per city (case-only -> full XGBoost):")
for city_name in sorted(report["models"]["xgb_full"]["test"]["per_city"].keys()):
    co = report["models"]["xgb_case_only"]["test"]["per_city"].get(city_name, {})
    fu = report["models"]["xgb_full"]["test"]["per_city"].get(city_name, {})
    if co and fu:
        mae_imp = co["MAE"] - fu["MAE"]
        mae_pct = mae_imp / co["MAE"] * 100 if co["MAE"] > 0 else 0
        rmse_imp = co["RMSE"] - fu["RMSE"]
        rmse_pct = rmse_imp / co["RMSE"] * 100 if co["RMSE"] > 0 else 0
        r2_imp = fu["R2"] - co["R2"]
        print(f"  {city_name:25s}  MAE: {co['MAE']:.1f} -> {fu['MAE']:.1f} ({mae_imp:+.1f}, {mae_pct:+.1f}%)  RMSE: {co['RMSE']:.1f} -> {fu['RMSE']:.1f} ({rmse_imp:+.1f}, {rmse_pct:+.1f}%)  R2: {co['R2']:.3f} -> {fu['R2']:.3f} (delta {r2_imp:+.3f})")

# Save per-city table
per_city_csv = OUTPUT_DIR / "per_city_results.csv"
per_city_df.to_csv(per_city_csv, index=False)
print(f"  Per-city results saved -> {per_city_csv}")

PER-CITY PERFORMANCE BREAKDOWN (2020–2021 Test Year)
MAE by City x Model:
Model                persistence  xgb_case_only  xgb_full
City                                                     
Cagayan de Oro City         6.50           6.40      6.30
Davao City                  7.90           7.40      7.00
Zamboanga City              3.60           4.30      4.40

RMSE by City x Model:
Model                persistence  xgb_case_only  xgb_full
City                                                     
Cagayan de Oro City         9.60           9.00      8.90
Davao City                 11.00          10.40     10.00
Zamboanga City              6.60           6.70      7.10

R2 by City x Model:
Model                persistence  xgb_case_only  xgb_full
City                                                     
Cagayan de Oro City        -0.28          -0.14     -0.11
Davao City                 -0.14          -0.03      0.05
Zamboanga City              0.30           0.28      0.19

Weather abl

In [11]:
# Ablation comparison: Case-only vs Full XGBoost
fig, ax = plt.subplots(figsize=(8, 5))
abl = report['ablation']
models_abl = ['Case-only XGBoost', 'Full XGBoost']
maes = [abl['case_only_MAE'], abl['full_MAE']]
ci_lo = [abl['case_only_ci']['MAE_ci'][0], abl['full_ci']['MAE_ci'][0]]
ci_hi = [abl['case_only_ci']['MAE_ci'][1], abl['full_ci']['MAE_ci'][1]]
err_lo = [m - lo for m, lo in zip(maes, ci_lo)]
err_hi = [hi - m for m, hi in zip(maes, ci_hi)]
bars = ax.bar(models_abl, maes, color=['#4e79a7', '#f28e2b'], width=0.5,
              yerr=[err_lo, err_hi], capsize=5, alpha=0.85)
for bar, mae in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'MAE={mae:.1f}', ha='center', fontsize=11)
ax.set_ylabel('MAE (cases)')
ax.set_title('Weather Ablation: Case-only vs Full Weather Features\n(2020–2021 Test Year, 95% bootstrap CI)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"MAE improvement from weather features: {abl['MAE_improvement']:+.1f} ({abl['MAE_improvement_pct']:+.1f}%)")
print(f"RMSE improvement: {abl['RMSE_improvement']:+.1f} ({abl['RMSE_improvement_pct']:+.1f}%)")

MAE improvement from weather features: +0.1 (+1.0%)
RMSE improvement: +0.1 (+1.0%)


In [12]:
# Per-year performance within the test set (2020 vs 2021)
# 2020 was a COVID-era anomaly; 2021 only has 2 weeks of data.
# The combined 2020–2021 test set is evaluated above.
print('No separate test period: 2020-2021 are both in the test set (combined).')
print('Pre-COVID models (trained on 2016–2019) may underperform on 2020 data.')


No separate test period: 2020-2021 are both in the test set (combined).
Pre-COVID models (trained on 2016–2019) may underperform on 2020 data.


In [13]:
# Actual vs predicted on the 2020–2021 test set, one line per city.
test_mask = splits['test']
city_colors = {'Davao City': '#4e79a7', 'Cagayan de Oro City': '#f28e2b',
               'Zamboanga City': '#59a14f'}
y_true_all = y.values[test_mask]
city_all = np.asarray(city.values)[test_mask]
date_all = pd.to_datetime(pd.Series(dates.values))[test_mask].values

fig, axes = plt.subplots(2, 2, figsize=(16, 9), squeeze=False)
flat = axes.flatten()
for i, result in enumerate(results):
    ax = flat[i]
    pred = np.asarray(result['predictions']['test'])
    for c in sorted(pd.unique(city_all)):
        cm = city_all == c
        col = city_colors.get(c)
        ax.plot(date_all[cm], y_true_all[cm], '-', color=col, linewidth=1.8,
                label=f'{c} - actual')
        ax.plot(date_all[cm], pred[cm], '--', color=col, linewidth=1.4,
                alpha=0.85, label=f'{c} - predicted')
    ax.set_title(f"{result['name']}  (MAE = {result['test_metrics']['MAE']:.2f})")
    ax.set_xlabel('Forecast-origin date')
    ax.set_ylabel('Dengue cases (t+2)')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
plt.suptitle('Actual vs Predicted Dengue Cases (2020–2021 Test Set)', fontsize=14)
plt.tight_layout()
plt.show()

## Step 8 — Final Report

All outputs are saved to `outputs/`.


In [14]:
# Final summary
print('All outputs saved to outputs/:')
print('  results.json                - full evaluation report')
print('  summary_table.csv           - model comparison table')
print('  per_city_results.csv        - per-city breakdown')
print('  data_validation_report.json - data quality checks')
print('  ablation_summary.json       - weather ablation results')
print('  figures/                    - all plots')

print('Key findings:')
abl = report['ablation']
best_mae = min(report['models'][m]['test']['pooled']['MAE'] for m in report['models'])
persist_mae = report['models']['persistence']['test']['pooled']['MAE']
print(f"  Weather features change MAE by {abl['MAE_improvement']:+.1f} "
      f"({abl['MAE_improvement_pct']:+.1f}%) vs case-only "
      f"(negative = weather features made it WORSE on this panel)")
print(f'  Best model MAE on 2020–2021 test: {best_mae:.1f} (vs persistence: {persist_mae:.1f})')

# Per-city direction of the weather effect
print('  Weather effect by city (full XGB vs case-only MAE):')
for c in sorted(report['models']['xgb_full']['test']['per_city'].keys()):
    co = report['models']['xgb_case_only']['test']['per_city'].get(c, {})
    fu = report['models']['xgb_full']['test']['per_city'].get(c, {})
    if co and fu:
        d = co['MAE'] - fu['MAE']
        verdict = 'weather HELPS' if d > 0 else 'weather HURTS'
        print(f"    {c:24s} {co['MAE']:6.2f} -> {fu['MAE']:6.2f} "
              f"({d:+.2f})  {verdict}")

print()
print('Statistical significance of these differences: run benchmark_wilcoxon.ipynb')


All outputs saved to outputs/:
  results.json                - full evaluation report
  summary_table.csv           - model comparison table
  per_city_results.csv        - per-city breakdown
  data_validation_report.json - data quality checks
  ablation_summary.json       - weather ablation results
  figures/                    - all plots
Key findings:
  Weather features change MAE by +0.1 (+1.0%) vs case-only (negative = weather features made it WORSE on this panel)
  Best model MAE on 2020–2021 test: 5.9 (vs persistence: 6.0)
  Weather effect by city (full XGB vs case-only MAE):
    Cagayan de Oro City        6.40 ->   6.32 (+0.07)  weather HELPS
    Davao City                 7.35 ->   7.05 (+0.30)  weather HELPS
    Zamboanga City             4.25 ->   4.45 (-0.20)  weather HURTS

Statistical significance of these differences: run benchmark_wilcoxon.ipynb
